# SLURM
The [Simple Linux Utility for Resource Management (SLURM)](https://slurm.schedmd.com) job scheduler is currently used on most HPC systems. Side note the company behind SLURM named [SchedMD was recently acquired by Nvidia](https://blogs.nvidia.com/blog/nvidia-acquires-schedmd/). Nevertheless, users primarily interact with it through three primary commands:

## `sbatch`
Submit a batch job script from the head node or login node of the high performance computer (HPC) to the job scheduler.

Typical usage:
```bash
sbatch job.sh
```

The script usually contains:

* resource requests (--nodes, --ntasks, --time, etc.)
* environment / module loading
* application launch commands

A list of all available options is available as part of the [SLURM documentation](https://slurm.schedmd.com/sbatch.html).

Example `job.sh` script:
```bash
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --error=error.out
#SBATCH --job-name=test
#SBATCH --chdir=/u/janj/slurmtest
#SBATCH --get-user-env=L
#SBATCH --partition=s.cmfe
#SBATCH --time=60
#SBATCH --ntasks=8

srun hostname
```

This example is going to return the `hostname` eight times, given the number of tasks is set to eight.

## `srun`
Launch parallel tasks inside an allocation.

Typical usage:

`srun -n 4 hostname`

In this case the command `hostname` is only executed four times. Consequently, one `sbatch` script can have multiple `srun` commands, which are either executed in serial or in parallel. For example:

```bash
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --error=error.out
#SBATCH --job-name=test
#SBATCH --chdir=/u/janj/slurmtest
#SBATCH --get-user-env=L
#SBATCH --partition=s.cmfe
#SBATCH --time=60
#SBATCH --ntasks=8

srun -n 4 hostname &
srun -n 4 hostname &
wait
```

Common use cases:

* launching MPI applications
* interactive execution
* starting multiple tasks within one allocation

srun is frequently used:

* inside batch scripts submitted with sbatch
* interactively after obtaining an allocation

Again, a list of all available options is available as part of the [SLURM documentation](https://slurm.schedmd.com/srun.html).

## `squeue`

Inspect the current scheduler queue.

Typical usage:

`squeue -u $USER`

Shows:

* pending jobs
* running jobs
* job IDs
* node assignments
* queue status

Further details are available as part of the [SLURM documentation](https://slurm.schedmd.com/squeue.html).

## `mpi4py`
The message passing interface (MPI) for Python is available in the [mpi4py](https://mpi4py.readthedocs.io) package. It exposes the MPI communicator launched by SLURM. A typical MPI Python script is started with `srun`:
```bash
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --error=error.out
#SBATCH --job-name=test
#SBATCH --chdir=/u/janj/slurmtest
#SBATCH --get-user-env=L
#SBATCH --partition=s.cmfe
#SBATCH --time=60
#SBATCH --ntasks=8

srun --mpi=pmix -n 8 python script.py
```
Depending on your version of MPI you might need to specify the MPI backend using `--mpi`.

Inside the Python script `script.py`:
```python
from mpi4py import MPI
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()
print(rank, size)
```

The result of this script is the list of task ranks and the total number of tasks:
```
2 8
0 8
4 8
3 8
6 8
5 8
1 8
7 8
```

If the script only returns eight entries of `0 1` then the MPI backend is not correctly configured.

### Option 1: Split the Allocation with Multiple srun Calls

```bash
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --error=error.out
#SBATCH --job-name=test
#SBATCH --chdir=/u/janj/slurmtest
#SBATCH --get-user-env=L
#SBATCH --partition=s.cmfe
#SBATCH --time=60
#SBATCH --ntasks=8

srun --mpi=pmix -n 4 python script.py &
srun --mpi=pmix -n 4 python script.py &
wait
```

This starts two independent MPI jobs inside the same SLURM allocation.

Each script sees its own `MPI.COMM_WORLD` and the resulting output is:

```
2 4
3 4
1 4
0 4
2 4
1 4
0 4
3 4
```

This is useful when launching separate applications or executables. However, many repeated srun calls can create scheduler/launcher overhead and may cause contention on some systems.

### Option 2: Use One srun and Split Communicators in Python

```bash
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --error=error.out
#SBATCH --job-name=test
#SBATCH --chdir=/u/janj/slurmtest
#SBATCH --get-user-env=L
#SBATCH --partition=s.cmfe
#SBATCH --time=60
#SBATCH --ntasks=8

srun --mpi=pmix -n 8 python script.py
```

Inside `script.py`:

```python
from mpi4py import MPI

def task(comm):
    rank = comm.Get_rank()
    size = comm.Get_size()
    print(rank, size)

world = MPI.COMM_WORLD
rank = world.Get_rank()
color = 0 if rank < 4 else 1
subcomm = world.Split(color=color, key=rank)
if color == 0:
    # ranks 0–3 run task A
    task(subcomm)
else:
    # ranks 4–7 run task B
    task(subcomm)
```

Now there is only one MPI launch. The Python script divides `MPI.COMM_WORLD` into smaller communicators.

Each task receives a communicator of size 4:

`subcomm.Get_size()`

Key Difference:
* Multiple srun calls create multiple independent MPI jobs.
* Splitting communicators creates one MPI job with multiple logical MPI groups inside it.
* Use multiple srun calls when you need separate programs, environments, or executables.
* Use communicator splitting when the workflow is controlled from one Python script and the tasks can share one MPI launch.